In [1]:
%%capture
!pip install torch

In [2]:
import json
import os
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader

import utaeT
from utils import iterate, overall_performance, save_results, prepare_output, pad_collate, get_ntrainparams, config_to_json, TreeDataset

In [3]:
@dataclass
class ConfigPaths:
    # Dataset / paths
    path_base: Path = Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "preprocessed_data"
    paths_sentinel_data: list = None
    paths_tree_data: list = None
    paths_date_data: list = None

    def __post_init__(self):
        self.paths_sentinel_data = [
            #self.path_base / "satellite_patches_bar.npy",
            self.path_base / "satellite_patches_ham.npy",
            self.path_base / "satellite_patches_par.npy",
            #self.path_base / "satellite_patches_vie.npy",
        ]

        self.paths_tree_data = [
            #self.path_base / "tree_patches_bar.npy",
            self.path_base / "tree_patches_ham.npy",
            self.path_base / "tree_patches_par.npy",
            #self.path_base / "tree_patches_vie.npy",
        ]

        self.paths_date_data = [
            #self.path_base / "date_patches_bar.npy",
            self.path_base / "date_patches_ham.npy",
            self.path_base / "date_patches_par.npy",
            #self.path_base / "date_patches_vie.npy",
        ]
    res_dir: Path = field(default_factory=lambda: Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "results_modelT")

In [4]:
@dataclass
class Config:
    # Model parameters
    input_dim = 5
    encoder_widths: list = field(default_factory=lambda: [32, 32, 64, 64])
    decoder_widths: list = field(default_factory=lambda: [16, 16, 32, 64])
    out_conv: int = 1

    str_conv_k: int = 4
    str_conv_s: int = 2
    str_conv_p: int = 1

    agg_mode: str = "att_group"
    encoder_norm: str = "group"

    n_head: int = 8
    d_model: int = 64
    d_k: int = 8

    num_workers: int = 4
    rdm_seed: int = 1
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)

    cache: bool = False

    # Training
    epochs: int = 1
    batch_size: int = 4
    lr: float = 0.001

    pad_value: float = 0.0
    padding_mode: str = "reflect"

    val_every: int = 1
    val_after: int = 0
    display_step: int = 1

    fold: int = 1

In [5]:
def weight_init(m):

    if isinstance(m, (nn.Conv2d, nn.Conv3d, nn.Conv1d,
                      nn.ConvTranspose2d, nn.ConvTranspose3d, nn.ConvTranspose1d)):

        init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

        if m.bias is not None:
            init.zeros_(m.bias)

    elif isinstance(m, nn.Linear):
        init.xavier_uniform_(m.weight)

        if m.bias is not None:
            init.zeros_(m.bias)

    elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm3d, nn.BatchNorm1d)):
        init.ones_(m.weight)
        init.zeros_(m.bias)

In [6]:
def checkpoint(fold, log, config_paths):
    # create checkpoints to save best model for validation
    with open(
        os.path.join(config_paths.res_dir, "Fold_{}".format(fold), "trainlog.json"), "w"
    ) as outfile:
        json.dump(log, outfile, indent=4)

In [7]:
def split_into_5_blocks(N):
    # split all indices of data set into 5 blocks for train/val/test split

    indices = np.arange(N)

    # split into 5 (nearly) equal parts
    blocks = np.array_split(indices, 5)

    return blocks


In [8]:
def make_fold_sequence(blocks):
    # create the train/val/test split fold sequence

    fold_sequence = []

    for i in range(len(blocks)):

        train = np.concatenate([
            blocks[(i + 0) % 5],
            blocks[(i + 1) % 5],
            blocks[(i + 2) % 5],
        ])

        val = blocks[(i + 3) % 5]
        test = blocks[(i + 4) % 5]

        fold_sequence.append([train, val, test])

    return fold_sequence

In [9]:
def main(config, config_paths):
    # set random seed and device
    np.random.seed(config.rdm_seed)
    torch.manual_seed(config.rdm_seed)
    device = torch.device(config.device)

    sentinel_list = []
    tree_list = []
    date_list = []
    
    for p_s, p_t, p_d in zip(config_paths.paths_sentinel_data,config_paths.paths_tree_data,config_paths.paths_date_data):
        sentinel_list.append(np.load(p_s, mmap_mode='r'))
        tree_list.append(np.load(p_t, mmap_mode='r'))
        date_list.append(np.load(p_d, mmap_mode='r'))

    city_blocks = []

    for s, t, d in zip(sentinel_list, tree_list, date_list):
        blocks = split_into_5_blocks(len(s))
        city_blocks.append((s, t, d, blocks))

    fold_sequence = []

    for fold_id in range(5):
    
        train_x, train_y, train_d = [], [], []
        val_x, val_y, val_d = [], [], []
        test_x, test_y, test_d = [], [], []
        for s, t, d, blocks in city_blocks:
            # TRAIN = 3 folds per city
            train_idx = np.concatenate([
                blocks[(fold_id + 0) % 5],
                blocks[(fold_id + 1) % 5],
                blocks[(fold_id + 2) % 5],
            ])
    
            val_idx = blocks[(fold_id + 3) % 5]
            test_idx = blocks[(fold_id + 4) % 5]
    
            train_x.append(s[train_idx])
            train_y.append(t[train_idx])
            train_d.append(d[train_idx])
    
            val_x.append(s[val_idx])
            val_y.append(t[val_idx])
            val_d.append(d[val_idx])
    
            test_x.append(s[test_idx])
            test_y.append(t[test_idx])
            test_d.append(d[test_idx])
    
        # merge all cities
        train_x = np.concatenate(train_x, axis=0)
        train_y = np.concatenate(train_y, axis=0)
        train_d = np.concatenate(train_d, axis=0)
    
        val_x = np.concatenate(val_x, axis=0)
        val_y = np.concatenate(val_y, axis=0)
        val_d = np.concatenate(val_d, axis=0)
    
        test_x = np.concatenate(test_x, axis=0)
        test_y = np.concatenate(test_y, axis=0)
        test_d = np.concatenate(test_d, axis=0)
    
        fold_sequence.append([
            (train_x, train_d, train_y),
            (val_x, val_d, val_y),
            (test_x, test_d, test_y),
        ])

    # create lists for test evaluation
    all_preds_folds = []
    all_targets_folds = []
    val_preds_folds = []
    val_targets_folds = []

    # define which folds should be trained
    if config.fold is not None:
        folds_to_run = [config.fold-1]
    else:
        folds_to_run = range(len(fold_sequence))
    
    # prepare output folders
    prepare_output(config_paths,config)

    for fold in folds_to_run:
        (train_x, train_d, train_y), (val_x, val_d, val_y), (test_x, test_d, test_y) = fold_sequence[fold]

        dt_train = TreeDataset(train_x, train_d, train_y)
        dt_val   = TreeDataset(val_x, val_d, val_y)
        dt_test  = TreeDataset(test_x, test_d, test_y)

        # create dataloaders using padding
        collate_fn = lambda x: pad_collate(x, pad_value=config.pad_value)
        train_loader = DataLoader(
            dt_train,
            batch_size=config.batch_size,
            shuffle=True,
            drop_last=True,
            collate_fn=collate_fn,
        )
        val_loader = DataLoader(
            dt_val,
            batch_size=config.batch_size,
            shuffle=False,
            drop_last=False,
            collate_fn=collate_fn,
        )
        test_loader = DataLoader(
            dt_test,
            batch_size=config.batch_size,
            shuffle=False,
            drop_last=False,
            collate_fn=collate_fn,
        )

        print(f"Train {len(dt_train)}, Val {len(dt_val)}, Test {len(dt_test)}")

        # Model definition
        model = utaeT.UTAE(config)

        # get number of parameters
        config.N_params = get_ntrainparams(model)

        # safe config for future reproducibility
        with open(os.path.join(config_paths.res_dir, "conf.json"), "w") as file:
            json.dump(config_to_json(config), file, indent=4)

        # print info about the model
        print(model)
        print("TOTAL TRAINABLE PARAMETERS :", config.N_params)
        print("Trainable layers:")
        for name, p in model.named_parameters():
            if p.requires_grad:
                print(name)

        # get model on right device and initialize weights
        model = model.to(device)
        model.apply(weight_init)

        # Optimizer and Loss
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
        criterion = nn.SmoothL1Loss()

        # Training loop
        trainlog = {}
        best_loss = float("inf")
        for epoch in range(1, config.epochs + 1):
            print(f"EPOCH {epoch}/{config.epochs}")

            model.train()
            train_metrics, _, _ = iterate(
                model,
                data_loader=train_loader,
                criterion=criterion,
                config=config,
                optimizer=optimizer,
                mode="train"
            )
            if epoch % config.val_every == 0 and epoch > config.val_after:
                print("Validation . . . ")
                model.eval()
                val_metrics, val_preds, val_targets = iterate(
                    model,
                    data_loader=val_loader,
                    criterion=criterion,
                    config=config,
                    optimizer=optimizer,
                    mode="val"
                )
                val_preds_folds.append(val_preds)
                val_targets_folds.append(val_targets)

                print(f"Loss: {val_metrics['val_loss']:.4f}")

                trainlog[epoch] = {**train_metrics, **val_metrics}
                checkpoint(fold + 1, trainlog, config_paths)
                if val_metrics["val_loss"] < best_loss:
                    best_loss = val_metrics["val_loss"]
                    torch.save(
                        {
                            "epoch": epoch,
                            "state_dict": model.state_dict(),
                            "optimizer": optimizer.state_dict(),
                        },
                        os.path.join(config_paths.res_dir, f"Fold_{fold + 1}","model.pth.tar"),
                    )
            else:
                trainlog[epoch] = {**train_metrics}
                checkpoint(fold + 1, trainlog, config_paths)

        print("Testing best epoch . . .")
        model.load_state_dict(
            torch.load(
                os.path.join(
                    config_paths.res_dir, f"Fold_{fold + 1}", "model.pth.tar"
                )
            )["state_dict"]
        )
        model.eval()

        test_metrics, test_preds, test_targets = iterate(
            model,
            data_loader=test_loader,
            criterion=criterion,
            config=config,
            optimizer=optimizer,
            mode="test",
        )
        all_preds_folds.extend(test_preds)
        all_targets_folds.extend(test_targets)
        print(f"Loss {test_metrics['test_loss']:.4f}")
        overall_performance(test_preds,test_targets,overall=False)
        save_results(fold + 1,test_metrics,config,config_paths,test_preds,test_targets)

    if config.fold is None:
        overall_performance(all_preds_folds, all_targets_folds)

In [10]:
if __name__ == "__main__":
    config = Config()
    config_paths = ConfigPaths()
    main(config, config_paths)

ValueError: mmap length is greater than file size